# TPC-AgentCF Colab Quickstart

This notebook is configured to avoid DeepSeek/OpenAI calls. It downloads and uses a small free local model from Hugging Face: `google/flan-t5-small`.

In [ ]:
import os
from pathlib import Path

REPO_URL = os.environ.get('TPC_AGENTCF_REPO_URL', 'https://github.com/YOUR-USERNAME/TPC-AgentCF.git')
REPO_DIR = Path('/content/TPC-AgentCF')

if not REPO_DIR.exists():
    if 'YOUR-USERNAME' in REPO_URL:
        raise ValueError('Set TPC_AGENTCF_REPO_URL or edit REPO_URL to your pushed GitHub repository before running this notebook.')
    !git clone "$REPO_URL" "$REPO_DIR"

%cd /content/TPC-AgentCF
print('Repo ready at', REPO_DIR)

In [ ]:
!python -m pip install --upgrade pip
!pip install "numpy<2" pytest transformers sentencepiece
!pip install -r requirements.txt

In [ ]:
import os
import yaml
from pathlib import Path

# Hard-disable remote paid endpoints for the notebook run.
for key in ['OPENAI_API_KEY', 'OPENAI_BASE_URL', 'OPENAI_MODEL']:
    os.environ.pop(key, None)

default_config_path = Path('config/default.yaml')
colab_config_path = Path('config/colab_localhf.yaml')

with default_config_path.open('r', encoding='utf-8') as handle:
    config = yaml.safe_load(handle)

config['llm']['backend'] = 'local_hf'
config['llm']['model'] = 'google/flan-t5-small'
config['llm']['max_tokens'] = 64
config['data']['dataset'] = 'movielens'
config['data']['max_users'] = 100
config['data']['max_items'] = 500
config['evaluation']['run_ablations'] = True

with colab_config_path.open('w', encoding='utf-8') as handle:
    yaml.safe_dump(config, handle, sort_keys=False)

print('Wrote', colab_config_path)
print(yaml.safe_dump(config, sort_keys=False))

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = 'google/flan-t5-small'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
print('Downloaded local model:', model_name)
print('Number of parameters:', sum(p.numel() for p in model.parameters()))

In [ ]:
!python -m pytest -q

In [ ]:
!python scripts/prepare_data.py --config config/colab_localhf.yaml

In [ ]:
!python scripts/run_baselines.py --config config/colab_localhf.yaml

In [ ]:
!python scripts/run_tpc_agentcf.py --config config/colab_localhf.yaml

In [ ]:
!python scripts/run_ablation.py --config config/colab_localhf.yaml

In [ ]:
!python scripts/make_paper_tables.py --config config/colab_localhf.yaml

In [ ]:
import pandas as pd

all_results = pd.read_csv('outputs/metrics/all_users/tpc_agentcf_results.csv')
conflict_results = pd.read_csv('outputs/metrics/conflict_users/tpc_agentcf_results.csv')
high_conflict_results = pd.read_csv('outputs/metrics/high_conflict_users/tpc_agentcf_results.csv')

display(all_results)
display(conflict_results)
display(high_conflict_results)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

recs = pd.read_json('outputs/explanations/recommendations.jsonl', lines=True)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
recs['conflict_score'].hist(ax=axes[0], bins=20)
axes[0].set_title('Conflict Score Distribution')
axes[0].set_xlabel('conflict_score')

summary = pd.read_csv('outputs/ablations/ablation_results.csv')
pivot = summary[summary['group'] == 'all_users'][['variant', 'MRR@10']].set_index('variant')
pivot.plot(kind='bar', ax=axes[1], legend=False)
axes[1].set_title('MRR@10 by Ablation Variant')
axes[1].set_ylabel('MRR@10')
plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path

for path in [
    'outputs/paper_assets/research_claims.md',
    'outputs/paper_assets/main_table.md',
    'outputs/paper_assets/qualitative_examples.md',
]:
    print('\n' + '=' * 80)
    print(path)
    print('=' * 80)
    print(Path(path).read_text(encoding='utf-8'))